# Inferential Statistical Analysis Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Inferential Statistical Analysis**.  
It demonstrates how we use a sample to draw conclusions about a population.

Main topics covered:

1. Population vs sample  
2. Point estimation  
3. Sampling distribution analysis  
4. Standard error  
5. Confidence intervals  
6. Margin of error  
7. Hypothesis testing  
8. Statistical significance testing  
9. Type I and Type II errors  
10. Small exercises and summary tables  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statistics import NormalDist

np.random.seed(42)

## 1. Create a Population and a Sample

Inferential statistics starts with the distinction between a **population** and a **sample**.

- The **population** is the full set of possible observations.
- The **sample** is the subset we actually observe.

Here we simulate a population and then draw a random sample from it.

In [ ]:
population = np.random.normal(loc=100, scale=15, size=10000)
sample = np.random.choice(population, size=64, replace=False)

population.mean(), population.std(ddof=0), sample.mean(), sample.std(ddof=1)

## 2. Point Estimation

A **point estimate** is a single numerical estimate of an unknown population parameter.

### Sample mean as an estimate of the population mean

$$
\hat{\mu} = \bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
$$

### Sample variance as an estimate of population variance

$$
\hat{\sigma}^2 = s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2
$$

In [ ]:
point_estimates = pd.DataFrame({
    'Quantity': ['Population Mean', 'Sample Mean', 'Population Variance', 'Sample Variance'],
    'Value': [population.mean(), sample.mean(), population.var(ddof=0), sample.var(ddof=1)]
})
point_estimates

## 3. Sampling Distribution of the Mean

A **sampling distribution** is the distribution of a statistic across many repeated samples.

For the sample mean:

$$
E[\bar{x}] = \mu
$$

$$
\operatorname{Var}(\bar{x}) = \frac{\sigma^2}{n}
$$

The standard deviation of the sampling distribution of the mean is called the **standard error**:

$$
SE(\bar{x}) = \frac{\sigma}{\sqrt{n}}
$$

In [ ]:
n = 64
means = []
for _ in range(1000):
    s = np.random.choice(population, size=n, replace=False)
    means.append(s.mean())

means = np.array(means)
means.mean(), means.std(ddof=1)

### Visualize the Sampling Distribution

If the sample size is large enough, the sampling distribution of the mean tends to look approximately normal.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(means, bins=25)
plt.title('Sampling Distribution of the Sample Mean')
plt.xlabel('Sample Mean')
plt.ylabel('Frequency')
plt.show()

## 4. Standard Error

The **standard error** measures how much a sample statistic varies from sample to sample.

For the sample mean:

$$
SE(\bar{x}) = \frac{\sigma}{\sqrt{n}}
$$

In practice, when \(\sigma\) is unknown, we often use:

$$
SE(\bar{x}) \approx \frac{s}{\sqrt{n}}
$$

In [ ]:
true_se = population.std(ddof=0) / np.sqrt(n)
estimated_se = sample.std(ddof=1) / np.sqrt(len(sample))
true_se, estimated_se

## 5. Confidence Intervals

A **confidence interval** provides a plausible range for the population parameter.

### 95% CI for a mean using a z critical value

$$
CI = \bar{x} \pm z_{\alpha/2}\frac{s}{\sqrt{n}}
$$

For a 95% confidence interval, we commonly use \(z = 1.96\).

In [ ]:
xbar = sample.mean()
s = sample.std(ddof=1)
n_sample = len(sample)
z_95 = 1.96

ci_lower = xbar - z_95 * (s / np.sqrt(n_sample))
ci_upper = xbar + z_95 * (s / np.sqrt(n_sample))
ci_lower, ci_upper

### Does the interval contain the true population mean?

In [ ]:
population_mean = population.mean()
population_mean, ci_lower <= population_mean <= ci_upper

## 6. Margin of Error

The **margin of error** is the half-width of the confidence interval.

$$
ME = z_{\alpha/2}\frac{s}{\sqrt{n}}
$$

A larger sample size usually reduces the margin of error.

In [ ]:
margin_of_error = z_95 * (s / np.sqrt(n_sample))
margin_of_error

### Compare Margin of Error for Different Sample Sizes

In [ ]:
sample_sizes = [16, 32, 64, 128, 256]
moe_values = []

for size in sample_sizes:
    temp_sample = np.random.choice(population, size=size, replace=False)
    temp_moe = z_95 * (temp_sample.std(ddof=1) / np.sqrt(size))
    moe_values.append(temp_moe)

pd.DataFrame({'Sample Size': sample_sizes, 'Margin of Error': moe_values})

## 7. Hypothesis Testing

A hypothesis test evaluates a claim about a population parameter.

Suppose the null hypothesis is:

$$
H_0: \mu = 100
$$

and the alternative is:

$$
H_1: \mu \neq 100
$$

A z-style test statistic is:

$$
z = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}
$$

In [ ]:
mu_0 = 100
z_stat = (xbar - mu_0) / (s / np.sqrt(n_sample))
z_stat

## 8. Statistical Significance Testing and p-Value

For a two-sided z-test, the p-value can be approximated using the standard normal distribution.

$$
p = 2\big(1-\Phi(|z|)\big)
$$

where \(\Phi\) is the standard normal cumulative distribution function.

In [ ]:
p_value = 2 * (1 - NormalDist().cdf(abs(z_stat)))
p_value

### Decision Rule

If \(p < 0.05\), we reject the null hypothesis at the 5% significance level.

In [ ]:
alpha = 0.05
decision = 'Reject H0' if p_value < alpha else 'Fail to reject H0'
decision

## 9. Two-Sample Comparison

Inferential statistics is also used to compare two groups.

Suppose we have results from two methods or two conditions.

In [ ]:
group_1 = np.random.normal(loc=102, scale=12, size=40)
group_2 = np.random.normal(loc=97, scale=12, size=40)

mean_1, mean_2 = group_1.mean(), group_2.mean()
std_1, std_2 = group_1.std(ddof=1), group_2.std(ddof=1)
n1, n2 = len(group_1), len(group_2)

z_two_sample = (mean_1 - mean_2) / np.sqrt((std_1**2 / n1) + (std_2**2 / n2))
p_two_sample = 2 * (1 - NormalDist().cdf(abs(z_two_sample)))

pd.DataFrame({
    'Group': ['Group 1', 'Group 2'],
    'Mean': [mean_1, mean_2],
    'Std': [std_1, std_2],
    'n': [n1, n2]
}), z_two_sample, p_two_sample

## 10. Type I and Type II Errors

In hypothesis testing:

- **Type I error**: reject a true null hypothesis  
- **Type II error**: fail to reject a false null hypothesis

Their probabilities are:

$$
P(\text{Type I Error}) = \alpha
$$

$$
P(\text{Type II Error}) = \beta
$$

Statistical power is:

$$
\text{Power} = 1 - \beta
$$

A simple intuition table:

In [ ]:
error_table = pd.DataFrame({
    'Reality': ['H0 is true', 'H0 is false'],
    'Reject H0': ['Type I Error', 'Correct Decision'],
    'Fail to Reject H0': ['Correct Decision', 'Type II Error']
})
error_table

## 11. Confidence Interval Coverage Simulation

A 95% confidence interval does not mean each interval has a 95% probability of containing the true mean.  
It means that across many repeated samples, about 95% of such intervals should contain the true mean.

In [ ]:
num_trials = 300
count_contains = 0

for _ in range(num_trials):
    s_temp = np.random.choice(population, size=50, replace=False)
    xbar_temp = s_temp.mean()
    s_temp_std = s_temp.std(ddof=1)
    me_temp = 1.96 * (s_temp_std / np.sqrt(len(s_temp)))
    lower = xbar_temp - me_temp
    upper = xbar_temp + me_temp
    if lower <= population_mean <= upper:
        count_contains += 1

coverage_rate = count_contains / num_trials
coverage_rate

## 12. Small Inferential Summary Table

This table summarizes several key inferential results from the current sample.

In [ ]:
summary_table = pd.DataFrame({
    'Quantity': [
        'Sample Mean',
        'Sample Std',
        'Estimated Standard Error',
        '95% CI Lower',
        '95% CI Upper',
        'Margin of Error',
        'z Statistic',
        'p Value'
    ],
    'Value': [
        xbar,
        s,
        estimated_se,
        ci_lower,
        ci_upper,
        margin_of_error,
        z_stat,
        p_value
    ]
})
summary_table

## 13. Mini Exercises

Try these on your own:

1. Change the sample size and observe how the standard error changes.  
2. Compute a 99% confidence interval instead of a 95% interval.  
3. Test a different null hypothesis, such as \(H_0: \mu = 98\).  
4. Compare two new groups and interpret the p-value.  
5. Repeat the coverage simulation with 1000 trials.  
6. Create an engineering-style example, such as displacement data or model RMSE data, and apply inferential tools.

These exercises are especially useful in AI, machine learning, structural engineering, and scientific data analysis.